# 05 · What drives demand (SHAP) and where should drivers go (dollar model)?

**Part A: SHAP.** An XGBoost model learns *when* each zone is busy relative to its own average (hour, weekday, month, holidays, toll zone, cluster, typical trip length). SHAP shows which factors matter most, and how demand timing differs between the toll zone and the rest of the city.

**Part B: dollar model.** For each cluster × time of day: how much longer riders waited in 2025 than in 2024, how many trips that puts at risk, and what serving them is worth in **platform margin** (base fare − driver pay; the toll and other fees pass through to the city/state). The key assumption, how many riders give up per extra minute of waiting, is shown as a **sensitivity range**.

In [ ]:
import os, sys
os.chdir(r"C:\Users\lalit\Documents\nyc-congestion-toll-impact")
sys.path.append("src")

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from xgboost import XGBRegressor
from pandas.tseries.holiday import USFederalHolidayCalendar
from zones import CBD_ZONES, BOUNDARY_ZONES

pd.set_option("display.width", 200)
CLUSTER_NAMES = {1: "Toll zone", 2: "Steady inner boroughs",
                 0: "Growing outer boroughs", 3: "Fast-growing periphery"}

## Part A · 1. Hourly demand panel (Uber, Feb–Dec 2025)

In [ ]:
zc = pd.read_csv("outputs/zone_clusters_uber.csv")
zc["cluster_name"] = zc["cluster"].map(CLUSTER_NAMES)

zone_miles = duckdb.sql("""
    SELECT pu_zone AS LocationID, SUM(sum_miles) / SUM(trips) AS avg_trip_miles
    FROM 'data/summary/*.parquet'
    WHERE company = 'Uber' AND trip_date BETWEEN '2024-02-01' AND '2024-12-31'
    GROUP BY 1
""").df()
zc = zc.merge(zone_miles, on="LocationID")

hourly = duckdb.sql("""
    SELECT pu_zone AS LocationID, trip_date, hour, SUM(trips) AS trips
    FROM 'data/summary/*.parquet'
    WHERE company = 'Uber' AND trip_date BETWEEN '2025-02-01' AND '2025-12-31'
    GROUP BY ALL
""").df()
hourly = hourly.merge(zc[["LocationID", "cluster", "area", "avg_trip_miles"]], on="LocationID")
hourly["trip_date"] = pd.to_datetime(hourly["trip_date"])
print(f"{len(hourly):,} zone-hour rows across {hourly['LocationID'].nunique()} zones")

## Part A · 2. Features and model

Target = log of trips relative to the zone's own average hour, so the model explains **timing** rather than just "big zones are big".

In [ ]:
hol = USFederalHolidayCalendar().holidays("2025-01-01", "2025-12-31")
AIRPORTS = [1, 132, 138]     # Newark, JFK, LaGuardia

h = hourly
h["y"] = np.log(h["trips"] / h.groupby("LocationID")["trips"].transform("mean"))
h["hour_of_day"] = h["hour"]
h["weekday"] = h["trip_date"].dt.dayofweek
h["is_weekend"] = (h["weekday"] >= 5).astype(int)
h["month"] = h["trip_date"].dt.month
h["is_holiday"] = h["trip_date"].isin(hol).astype(int)
h["toll_zone"] = h["LocationID"].isin(CBD_ZONES).astype(int)
h["airport"] = h["LocationID"].isin(AIRPORTS).astype(int)
h["borough"] = h["area"].map({"Toll zone": 0, "Boundary": 1, "Manhattan": 2, "Brooklyn": 3,
                              "Queens": 4, "Bronx": 5, "Staten Island": 6, "EWR": 7}).fillna(8)

FEATURES = ["hour_of_day", "weekday", "is_weekend", "month", "is_holiday",
            "toll_zone", "airport", "borough", "cluster", "avg_trip_miles"]

sample = h.sample(min(len(h), 600_000), random_state=42)
train = sample.sample(frac=0.8, random_state=1)
test = sample.drop(train.index)

model = XGBRegressor(n_estimators=400, learning_rate=0.08, max_depth=6,
                     subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
model.fit(train[FEATURES], train["y"])

pred = model.predict(test[FEATURES])
r2 = 1 - ((test["y"] - pred) ** 2).sum() / ((test["y"] - test["y"].mean()) ** 2).sum()
print(f"Test R² = {r2:.3f}  (share of hour-to-hour demand swings the model explains)")

## Part A · 3. SHAP: what drives demand timing?

In [ ]:
X_shap = test[FEATURES].sample(4000, random_state=7)
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(X_shap)

importance = pd.Series(np.abs(sv).mean(axis=0), index=FEATURES).sort_values(ascending=False)
importance.round(3).to_csv("outputs/shap_importance.csv")

shap.summary_plot(sv, X_shap, show=False, plot_size=(10, 6))
plt.title("What drives Uber demand timing (SHAP)")
plt.tight_layout()
plt.savefig("outputs/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
importance.round(3)

In [ ]:
# how the hourly demand curve differs: toll zone vs everywhere else
fig, ax = plt.subplots(figsize=(11, 5))
hour_idx = X_shap.columns.get_loc("hour_of_day")
for flag, label, color in [(1, "Toll zone", "#d62728"), (0, "Rest of city", "#1f77b4")]:
    mask = (X_shap["toll_zone"] == flag).values
    curve = pd.Series(sv[mask, hour_idx], index=X_shap["hour_of_day"].values[mask]).groupby(level=0).mean()
    ax.plot(curve.index, curve.values, "o-", color=color, label=label)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hour of day")
ax.set_ylabel("SHAP effect of hour on demand (log scale)")
ax.set_title("When demand peaks: toll zone vs rest of city")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/shap_hour_curve.png", dpi=150)
plt.show()

## Part B · 1. Wait times, margin and driver pay by cluster × time of day

In [ ]:
ids = ",".join(map(str, zc["LocationID"]))
blocks = duckdb.sql(f"""
    SELECT pu_zone AS LocationID,
           CASE WHEN trip_date < DATE '2025-01-01' THEN '2024' ELSE '2025' END AS yr,
           CASE WHEN hour < 6 THEN '1 Night (0-5)'
                WHEN hour < 10 THEN '2 Morning (6-9)'
                WHEN hour < 16 THEN '3 Midday (10-15)'
                WHEN hour < 20 THEN '4 Evening (16-19)'
                ELSE '5 Late (20-23)' END AS time_block,
           SUM(trips) AS trips,
           SUM(sum_base_fare) AS base_fare,
           SUM(sum_driver_pay) AS driver_pay,
           SUM(sum_trip_seconds) AS secs,
           SUM(sum_wait_seconds) AS wait_s,
           SUM(wait_trips) AS wait_n
    FROM 'data/summary/*.parquet'
    WHERE company = 'Uber' AND pu_zone IN ({ids})
      AND ((trip_date BETWEEN '2024-02-01' AND '2024-12-31')
        OR (trip_date BETWEEN '2025-02-01' AND '2025-12-31'))
    GROUP BY ALL
""").df().merge(zc[["LocationID", "cluster"]], on="LocationID")

cb = blocks.groupby(["cluster", "time_block", "yr"])[["trips", "base_fare", "driver_pay",
                                                        "secs", "wait_s", "wait_n"]].sum()
cb = cb.unstack("yr")
t = pd.DataFrame(index=cb.index)
t["trips_2025_K"] = cb[("trips", "2025")] / 1e3
t["trips_chg_pct"] = (cb[("trips", "2025")] / cb[("trips", "2024")] - 1) * 100
t["wait_2025_min"] = cb[("wait_s", "2025")] / cb[("wait_n", "2025")] / 60
t["wait_chg_min"] = t["wait_2025_min"] - cb[("wait_s", "2024")] / cb[("wait_n", "2024")] / 60
t["margin_per_trip"] = (cb[("base_fare", "2025")] - cb[("driver_pay", "2025")]) / cb[("trips", "2025")]
t["driver_pay_hr"] = cb[("driver_pay", "2025")] / (cb[("secs", "2025")] / 3600)
t = t.reset_index()
t.insert(1, "cluster_name", t["cluster"].map(CLUSTER_NAMES))
t.round(2)

## Part B · 2. Dollar model: trips at risk from longer waits

For every cluster × time block where waits **rose**, trips at risk = 2025 trips × extra wait (min) × *loss rate per extra minute*. Value = trips at risk × margin per trip.

The loss rate is **an assumption, not something the data measures** (riders who gave up never appear in trip records). So it's shown at three levels: 2%, 5% and 8% of riders lost per extra minute of wait. All figures are annualized from 11 months (Feb–Dec 2025).

In [ ]:
LOSS_RATES = {"low (2%/min)": 0.02, "mid (5%/min)": 0.05, "high (8%/min)": 0.08}
ANNUALIZE = 12 / 11

opp = t[t["wait_chg_min"] > 0].copy()
for name, rate in LOSS_RATES.items():
    at_risk = opp["trips_2025_K"] * 1e3 * opp["wait_chg_min"] * rate * ANNUALIZE
    opp[f"trips_at_risk_K {name}"] = at_risk / 1e3
    opp[f"margin_$M {name}"] = at_risk * opp["margin_per_trip"] / 1e6

opp = opp.sort_values("margin_$M mid (5%/min)", ascending=False)
opp.round(2).to_csv("outputs/driver_positioning_opportunity.csv", index=False)

cols = ["cluster_name", "time_block", "trips_2025_K", "wait_chg_min", "margin_per_trip",
        "driver_pay_hr", "trips_at_risk_K mid (5%/min)", "margin_$M mid (5%/min)"]
opp[cols].head(10).round(2)

In [ ]:
# totals by cluster at each assumption
summary = opp.groupby("cluster_name")[[c for c in opp.columns if c.startswith("margin_$M")]].sum()
summary.loc["TOTAL"] = summary.sum()
summary.round(2)